# Tiered KV RunPod H100 Benchmark

RunPod profile for one H100 using `runpod/pytorch:2.4.0-py3.11-cuda12.4.1-devel-ubuntu22.04`. This uses the same benchmark runner as local validation but larger context lengths and `Qwen/Qwen2.5-7B-Instruct` by default.

In [ ]:
# Run this cell first, then restart the kernel if packages were installed.
import os
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')

%pip install -q -r requirements.txt

import sys
print('python:', sys.version)
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES'))

In [ ]:
import json
import os
from pathlib import Path

import torch
from experiments.kv_benchmark import get_profile, run_benchmark

PROFILE = get_profile('runpod-h100', model_id=os.environ.get('MODEL_ID') or None)
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

POLICY_CONFIG_FILE = Path(os.environ.get('POLICY_CONFIG_FILE', 'autoresearch/best_local_policy_config.json'))
if not POLICY_CONFIG_FILE.exists() and os.environ.get('ALLOW_DEFAULT_POLICY') != '1':
    raise FileNotFoundError(
        f'Missing tuned policy config: {POLICY_CONFIG_FILE}. '
        'Copy autoresearch/best_local_policy_config.json to the pod, set POLICY_CONFIG_FILE, '
        'or set ALLOW_DEFAULT_POLICY=1 to intentionally run the untuned default policy.'
    )
POLICY_CONFIG = json.loads(POLICY_CONFIG_FILE.read_text()) if POLICY_CONFIG_FILE.exists() else None

print('profile:', PROFILE)
print('device:', DEVICE)
print('policy_config_file:', POLICY_CONFIG_FILE if POLICY_CONFIG else 'UNTUNED DEFAULT POLICY')
if POLICY_CONFIG:
    print(json.dumps(POLICY_CONFIG, indent=2))
if torch.cuda.is_available():
    print('visible cuda devices:', torch.cuda.device_count())
    print('gpu:', torch.cuda.get_device_name(0))


## Run H100 Benchmark

This is the shareable benchmark profile. It runs recall, perplexity, measured cache bytes, and throughput. NIAH remains optional because it adds time and variance.

In [ ]:
result = run_benchmark(
    PROFILE,
    device=DEVICE,
    policy_config=POLICY_CONFIG,
    run_recall=True,
    run_perplexity=True,
    run_niah=False,
    run_memory=True,
    output_dir=Path('results/runpod_h100'),
)

print('saved:', result.get('output_path'))
print(json.dumps(result['sanity'], indent=2))


## Tables

In [ ]:
import pandas as pd

recall_rows = []
for group in result.get('recall', []):
    for length, acc, ok, total in zip(group['lengths'], group['accuracy'], group['n_correct'], group['n_total']):
        recall_rows.append({
            'strategy': group['strategy'],
            'context_len': length,
            'accuracy': acc,
            'n_correct': ok,
            'n_total': total,
        })
recall_df = pd.DataFrame(recall_rows)
display(recall_df)

ppl_rows = []
for group in result.get('perplexity', []):
    for row in group['rows']:
        ppl_rows.append({'strategy': group['strategy'], **row})
ppl_df = pd.DataFrame(ppl_rows)
display(ppl_df)

mem_df = pd.DataFrame(result.get('memory', []))
thru_df = pd.DataFrame(result.get('throughput', []))
if not mem_df.empty:
    baseline_bytes = float(mem_df.loc[mem_df['strategy'] == 'FP16 baseline', 'cache_bytes_empirical'].iloc[0])
    mem_df['cache_MB'] = mem_df['cache_bytes_empirical'] / 1e6
    mem_df['ratio_to_baseline'] = mem_df['cache_bytes_empirical'] / baseline_bytes
    display(mem_df[['strategy', 'seq_len', 'cache_MB', 'ratio_to_baseline', 'peak_alloc_bytes', 'peak_reserved_bytes']])
if not thru_df.empty:
    display(thru_df[['strategy', 'tokens_per_second', 'median_seconds', 'n_new_tokens']])

## Plots

In [ ]:
import matplotlib.pyplot as plt

if not recall_df.empty:
    fig, ax = plt.subplots(figsize=(7, 4))
    for name, g in recall_df.groupby('strategy'):
        ax.plot(g['context_len'], g['accuracy'], marker='o', label=name)
    ax.set_xscale('log')
    ax.set_xlabel('context length')
    ax.set_ylabel('recall accuracy')
    ax.set_title('Synthetic key-value recall')
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()

if not ppl_df.empty:
    fig, ax = plt.subplots(figsize=(7, 4))
    for name, g in ppl_df.groupby('strategy'):
        ax.plot(g['context_len'], g['perplexity'], marker='o', label=name)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('context length')
    ax.set_ylabel('perplexity')
    ax.set_title('Sliding-window perplexity')
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()

if not mem_df.empty:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(mem_df['strategy'], mem_df['ratio_to_baseline'])
    ax.axhline(1.0, color='black', linestyle='--', linewidth=1)
    ax.set_ylabel('cache bytes / FP16 baseline')
    ax.set_title('Measured KV-cache memory')
    ax.tick_params(axis='x', rotation=20)
    plt.show()

## Optional NIAH Heatmap

In [ ]:
# Optional full H100 NIAH run.
# niah_result = run_benchmark(
#     PROFILE,
#     device=DEVICE,
#     policy_config=POLICY_CONFIG,
#     run_recall=False,
#     run_perplexity=False,
#     run_niah=True,
#     run_memory=False,
#     output_dir=Path('results/runpod_h100_niah'),
# )
# print(niah_result.get('output_path'))
